# Fast path — inverted-U checkpoint (v0)

**Gameplan:** `current_documents/sports_documents/SPORTS_DATA_GAMEPLAN.md` → *Fast path — first inverted-U checkpoint*.

**v0 rules:** dirty OK; do not write under `datasets/mbb/DO_NOT_ERASE/` except read. Outputs (figure + binned CSV) go to `datasets/mbb/exports_inverted_u_v0/`.

**Data:** Uses existing `player_season_panel_530.csv` (already has `Y_draft`, `perf`, `poolq_loo`, SR BPM merge).

In [ ]:
# v0 definitions — edit if you change the run
from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(".").resolve()
PANEL = PROJECT_ROOT / "datasets" / "mbb" / "player_season_panel_530.csv"
OUT_DIR = PROJECT_ROOT / "datasets" / "mbb" / "exports_inverted_u_v0"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = date.today().isoformat()

# Grain: player-season rows; PERF for LOO pool is column perf (same as 530 export)
MIN_MINUTES_V0 = 0  # tighten later, e.g. 100
VENTILES = 20

print("PANEL:", PANEL)
print("Exists:", PANEL.is_file())
print("RUN_STAMP:", RUN_STAMP)

In [ ]:
df = pd.read_csv(PANEL, low_memory=False)
print("rows", len(df), "cols", list(df.columns))

# v0 sample: drop missing LOO / outcome
use = df.dropna(subset=["poolq_loo", "Y_draft"]).copy()
if MIN_MINUTES_V0 > 0:
    use = use.loc[use["minutes"] >= MIN_MINUTES_V0]
print("after dropna(minutes if any)", len(use))
print("Y_draft mean", use["Y_draft"].mean())
print("perf (LOO base) describe:\n", use["perf"].describe())

In [ ]:
# Ventiles on poolq_loo (duplicates='drop' if many ties)
try:
    use["vent"] = pd.qcut(use["poolq_loo"], q=VENTILES, labels=False, duplicates="drop")
except ValueError as e:
    raise SystemExit(f"qcut failed: {e} — try fewer ventiles or rank-based bins") from e

bin_tab = (
    use.groupby("vent", observed=True)
    .agg(
        n=("Y_draft", "size"),
        draft_rate=("Y_draft", "mean"),
        poolq_mean=("poolq_loo", "mean"),
    )
    .reset_index()
    .sort_values("vent")
)
csv_path = OUT_DIR / f"binned_draft_rate_ventiles_{RUN_STAMP}.csv"
bin_tab.to_csv(csv_path, index=False)
print("Wrote", csv_path)
bin_tab.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bin_tab["poolq_mean"], bin_tab["draft_rate"], marker="o", ms=4)
ax.set_xlabel("Mean pool quality (poolq_loo) within ventile")
ax.set_ylabel("Draft rate (mean Y_draft)")
ax.set_title(f"v0 ventiles={VENTILES} (n≈{len(use):,} player-seasons)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
png = OUT_DIR / f"inverted_u_ventiles_{RUN_STAMP}.png"
fig.savefig(png, dpi=150)
print("Wrote", png)
plt.show()

## Optional: dirty LPM (sign check only)

$Y_{draft} \sim poolq\_loo + poolq\_sq$. For cluster SE, wire `linearmodels` or similar later.

In [ ]:
try:
    import statsmodels.api as sm
except ImportError:
    print("statsmodels not installed — skip LPM cell")
else:
    y = use["Y_draft"].astype(float)
    X = use[["poolq_loo", "poolq_sq"]].astype(float)
    X = sm.add_constant(X)
    ols = sm.OLS(y, X, missing="drop").fit()
    print(ols.summary())
    tplp = OUT_DIR / f"lpm_ols_summary_{RUN_STAMP}.txt"
    with open(tplp, "w") as f:
        f.write(ols.summary().as_text())
    print("Wrote", tplp)